# EDA Part 2 — Solutions: Histograms, Boxplots, Pairplots and Heatmaps

**Module 9: Exploratory Data Analysis**

> These are the reference solutions.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
## CHANGE PATH TO YOUR OWN
df = pd.read_csv('content/5-Data_Visualization_&_Storytelling/3-Visualization-best-practices/Superstore.csv', encoding='latin1')

df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

print(f'Shape: {df.shape}')
df.head(3)

Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


---
## Block 1 — Histograms

### Exercise 1.1 — Profit histogram with mean and median

In [ ]:
mean_profit   = df['Profit'].mean()
median_profit = df['Profit'].median()

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(df['Profit'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(mean_profit,   color='red',    linestyle='--', linewidth=1.8,
           label=f'Mean: ${mean_profit:.1f}')
ax.axvline(median_profit, color='orange', linestyle='--', linewidth=1.8,
           label=f'Median: ${median_profit:.1f}')

ax.set_title('Profit Distribution — Superstore', fontsize=13)
ax.set_xlabel('Profit ($)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

pct_loss = (df['Profit'] < 0).mean() * 100
print(f'Loss-making transactions: {pct_loss:.1f}%')

**Observations:**
- The distribution is **heavily right-skewed**: most transactions have low or moderate profit, but a few have very high gains.
- The **mean ($28) is well above the median ($9)**, confirming the skew — large profits pull the mean upward.
- Approximately **18% of transactions generate losses** — a relevant figure for the business.

### Exercise 1.2 — Discount Histograms by Category

In [ ]:
categories = df['Category'].unique()
colors = ['#2196F3', '#FF9800', '#4CAF50']

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for i, (cat, color) in enumerate(zip(categories, colors)):
    subset = df[df['Category'] == cat]['Discount']
    axes[i].hist(subset, bins=20, color=color, edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{cat}\nMedia: {subset.mean():.2f}')
    axes[i].set_xlabel('Discount')
    if i == 0:
        axes[i].set_ylabel('Frequency')

plt.suptitle('Discount Distribution by Category', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Observations:**
- All three categories show very similar distributions — most transactions have a discount of 0.
- There are peaks at discrete values (0.2, 0.3, 0.4...) because discounts are applied in round numbers.
- **Furniture** has a slightly longer tail towards high discounts, which may explain its lower profitability.

### Exercise 1.3 — Overlapping histogram by Region

In [ ]:
regions = df['Region'].unique()
colors_reg = ['#E91E63', '#3F51B5', '#009688', '#FF5722']

fig, ax = plt.subplots(figsize=(10, 5))

for region, color in zip(regions, colors_reg):
    subset = df[df['Region'] == region]['Sales']
    ax.hist(subset, bins=40, alpha=0.5, color=color,
            label=f'{region} (n={len(subset)})', edgecolor='white')

ax.set_title('Sales Distribution by Region', fontsize=13)
ax.set_xlabel('Sales ($)')
ax.set_ylabel('Frequency')
ax.set_xlim(0, 3000)  # Zoom in on the central range
ax.legend()
plt.tight_layout()
plt.show()

**Observations:**
- All 4 regions have very similar distributions — right-skewed, concentrated at low sales values.
- **West** tends to have slightly more volume (more rows) than South.
- The overlapping chart works here because the distributions are similar. If they were very different, the overlaps would make it hard to read — in that case, subplots would be better.

---
## Block 2 — Boxplots

### Exercise 2.1 — Profit boxplot by Segment

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.boxplot(data=df, x='Segment', y='Profit', palette='Set2', ax=ax)
ax.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Break-even point')
ax.set_ylim(-400, 600)
ax.set_title('Profit by Customer Segment', fontsize=13)
ax.set_xlabel('Segment')
ax.set_ylabel('Profit ($)')
ax.legend()
plt.tight_layout()
plt.show()

print(df.groupby('Segment')['Profit'].median().round(2))

**Observations:**
- **Home Office** has the highest Profit median — its sales tend to be more profitable.
- **Consumer** is the most variable segment (largest box) and has the most negative outliers.
- All three segments have positive medians, which is a good sign — the problem is not systemic but isolated.

### Exercise 2.2 — Profit boxplot for Furniture by Region

In [ ]:
furniture = df[df['Category'] == 'Furniture']

fig, ax = plt.subplots(figsize=(9, 5))

sns.boxplot(data=furniture, x='Region', y='Profit', palette='Set1', ax=ax)
ax.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Break-even point')
ax.set_ylim(-400, 400)
ax.set_title('Profit in Furniture by Region', fontsize=13)
ax.set_xlabel('Region')
ax.set_ylabel('Profit ($)')
ax.legend()
plt.tight_layout()
plt.show()

print(furniture.groupby('Region')['Profit'].median().round(2))

**Observations:**
- **Central** is the region with the lowest Profit median in Furniture — even negative.
- **East and West** perform better, with positive medians.
- Hypothesis: discounts in Furniture may be concentrated in the Central region. We would need to cross Discount × Region × Category to confirm.

### Exercise 2.3 — Multidimensional boxplot: Discount by Category and Segment

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

sns.boxplot(
    data=df,
    x='Category',
    y='Discount',
    hue='Segment',
    palette='Set2',
    ax=ax
)

ax.set_title('Applied Discount by Category and Segment', fontsize=13)
ax.set_xlabel('Category')
ax.set_ylabel('Discount')
ax.legend(title='Segment', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

**Observations:**
- Discounts are quite similar across segments within each category — there does not appear to be a segment-specific discount policy.
- **Furniture** has slightly higher discount medians than the other categories across all segments.
- Variability (box size) is similar across all combinations.

---
## Block 3 — Pairplot

### Exercise 3.1 — Basic pairplot

In [ ]:
cols = ['Sales', 'Quantity', 'Discount', 'Profit']
df_sample = df[cols].sample(600, random_state=42)

sns.pairplot(
    df_sample,
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 'color': 'steelblue'}
)
plt.suptitle('Pairplot — Superstore Numerical Variables', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**Observations:**
- **Discount vs Profit** shows the most striking relationship: a clear negative trend — the higher the discount, the lower (or negative) the profit.
- **Sales vs Profit** has a positive correlation but with a lot of dispersion — not all large orders are profitable.
- **Sales and Quantity** have heavily right-skewed distributions (long tail).
- **Quantity** does not appear to have a strong relationship with any other variable.

### Exercise 3.2 — Pairplot with hue by Category

In [ ]:
cols_cat = ['Sales', 'Quantity', 'Discount', 'Profit', 'Category']
df_sample_cat = df[cols_cat].sample(600, random_state=42)

sns.pairplot(
    df_sample_cat,
    hue='Category',
    diag_kind='kde',
    plot_kws={'alpha': 0.35},
    palette='Set2'
)
plt.suptitle('Pairplot by Category', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**Observations:**
- In the **Discount vs Profit** scatter, all three categories show the same negative trend — discounts destroy margin in all of them.
- **Technology** (Sales diagonal) has a longer tail — there are tech sales with much higher values.
- In the **Sales vs Profit** scatter, Technology has points more dispersed and higher up — suggesting higher margins but also more variability.
- New hypothesis: is the gross margin (%) different between categories, or only the volume?

---
## Block 4 — Correlation Heatmap

### Exercise 4.1 — Basic correlation heatmap

In [ ]:
cols_num = ['Sales', 'Quantity', 'Discount', 'Profit']
corr = df[cols_num].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5
)
plt.title('Correlation Matrix — Superstore', fontsize=13)
plt.tight_layout()
plt.show()

print('Correlations with Profit:')
print(corr['Profit'].drop('Profit').sort_values(key=abs, ascending=False).round(3))

**Observations:**
- **Discount** has the strongest correlation with Profit (negative: ~-0.22) — confirming what we saw in the pairplot.
- **Sales** has a moderate positive correlation with Profit (~+0.48).
- **Quantity** has very weak correlation with everything — it is not a profitability predictor on its own.
- No correlation is very high — profit depends on a combination of factors, not a single variable.

### Exercise 4.2 — Correlation heatmaps by Category

In [ ]:
categories = df['Category'].unique()
cols_corr = ['Sales', 'Discount', 'Profit']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, cat in enumerate(categories):
    subset = df[df['Category'] == cat][cols_corr]
    corr_cat = subset.corr()
    sns.heatmap(
        corr_cat,
        annot=True,
        fmt='.2f',
        cmap='coolwarm',
        vmin=-1, vmax=1,
        square=True,
        linewidths=0.5,
        ax=axes[i],
        cbar=(i == 2)  # Only show colorbar on the last one
    )
    axes[i].set_title(cat, fontsize=12, fontweight='bold')

plt.suptitle('Correlations by Category (Sales, Discount, Profit)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Observations:**
- The **Discount vs Profit correlation is negative in all three categories**, but stronger in **Furniture** and **Office Supplies** than in Technology.
- In **Technology**, Sales and Profit have a stronger positive correlation — large tech sales tend to be profitable.
- Conclusion: discounts destroy margin in all categories, but the effect is especially pronounced in Furniture.

---
## Block 5 — Integrated exercise

### Full Solution

In [ ]:
# Step 1 — Discount histogram
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['Discount'], bins=25, color='steelblue', edgecolor='white')
ax.set_title('Distribution of Applied Discounts')
ax.set_xlabel('Discount')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

print(f"Transacciones sin descuento: {(df['Discount'] == 0).mean()*100:.1f}%")
print(f"Transacciones con descuento > 0.4: {(df['Discount'] > 0.4).mean()*100:.1f}%")

In [ ]:
# Step 2 — Discount boxplot by Sub-Category (horizontal)
order = df.groupby('Sub-Category')['Discount'].median().sort_values(ascending=False).index

plt.figure(figsize=(10, 7))
sns.boxplot(
    data=df,
    y='Sub-Category',
    x='Discount',
    order=order,
    palette='Blues_r'
)
plt.title('Discount by Sub-Category (sorted by median)', fontsize=13)
plt.xlabel('Discount')
plt.tight_layout()
plt.show()

In [ ]:
# Step 3 — Discount vs Profit scatter with hue=Category
plt.figure(figsize=(10, 5))
sns.scatterplot(
    data=df,
    x='Discount',
    y='Profit',
    hue='Category',
    alpha=0.35,
    palette='Set2'
)
plt.axhline(0, color='red', linestyle='--', alpha=0.6)
plt.title('Discount vs Profit by Category', fontsize=13)
plt.xlabel('Applied discount')
plt.ylabel('Profit ($)')
plt.legend(title='Category')
plt.tight_layout()
plt.show()

In [ ]:
# Step 4 — Hypothesis and conclusions

# Main hypothesis:
# High discounts (>0.4) are destroying margin in certain sub-categories,
# especially in Furniture (Tables, Bookcases) and some Office Supplies.

# Evidence found:
# - The histogram shows that most transactions have discount=0,
#   but there is a relevant subset with discounts >0.3.
# - The sub-category boxplot reveals that Tables and Bookcases have
#   the highest discount medians.
# - The Discount vs Profit scatter confirms that with discount > 0.4,
#   virtually all transactions are losses.

# Questions to investigate next:
# 1. What is the discount threshold above which profit typically becomes negative?
# 2. Is there a geographical pattern in high discounts (Central region)?
# 3. Are high discounts applied more to certain customer segments?

# Quick check of the critical discount threshold:
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.9]
df['disc_bin'] = pd.cut(df['Discount'], bins=bins)
summary = df.groupby('disc_bin', observed=True)['Profit'].agg(['mean', 'median', lambda x: (x<0).mean()*100])
summary.columns = ['Mean Profit', 'Median Profit', '% Losses']
print(summary.round(1))